# LG-05: 并行执行与子图
> **阶段**: LG-05 | **难度**: 中高级 | **预计时长**: 5-6 小时 | **依赖**: LG-01, LG-02, LG-03

## 学习目标
- 掌握 Fan-out / Fan-in 模式实现并行节点执行
- 理解 Reducer 机制处理并发状态更新冲突
- 掌握 Send 原语实现动态 Map-Reduce
- 掌握 Subgraph（子图定义、调用、共享状态）
- 理解输入/输出映射配置
- 掌握并行模式最佳实践
- 了解并发控制与超时处理

In [ ]:
# 安装依赖（首次运行）
# !pip install -U langgraph langchain langchain-openai

---

## 开场引入：从「等待焦虑」引入

假设你要写一份行业研究报告。需要做四件事：查财报数据、查新闻舆情、查竞品动态、查专家观点。如果你一个人做，要一件一件来——做完第一件再做第二件，总时间是 4 个任务之和。

但如果你有一个团队：你负责统筹，4 个研究员同时分头查，你在汇总点等他们都完成后整合报告。总时间就是最慢的那个研究员的时间。

这就是并行执行的意义。今天的主题是：**让图学会「分工合作」**。

### 餐厅后厨的类比

你去餐厅点了一份套餐：主菜、配菜、汤、饮料。后厨不会等主菜做完了才做配菜——四个厨师同时开工，各自负责一道，然后在「出餐台」汇聚成一份完整的套餐端给你。

并行节点就是这四个厨师，Fan-in 汇聚点就是出餐台。

---

## 1. Fan-out / Fan-in 并行模式

### 1.1 什么是 Fan-out / Fan-in？

- **Fan-out（扇出）**：一个节点同时触发多个下游节点并行执行
- **Fan-in（扇入）**：多个节点都完成后，汇聚到一个节点

```
           +-→ [查财报] --+
[主题输入] --+--→ [查新闻] --+--→ [汇总报告] → END
           +--→ [查竞品] --+
           +--→ [查专家] --+
            ^ Fan-out      ^ Fan-in
```

**类比**：Fan-out 像是一个项目经理同时给 4 个人发邮件分配任务；Fan-in 像是项目经理在 Excel 里设置了条件格式——等 4 个人都回复了，这行才变绿。

### 1.2 基础并行图示例

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
import time

class ResearchState(TypedDict):
    topic: str
    finance_data: str
    news_sentiment: str
    competitor_info: str
    expert_opinions: str
    report: str
    sources: Annotated[list, add]

def collect_finance(state: ResearchState) -> dict:
    print("[collect_finance] 抓取财经数据...")
    time.sleep(0.3)
    return {"finance_data": "营收增长 25%，净利润率 18%", "sources": ["finance_report"]}

def analyze_sentiment(state: ResearchState) -> dict:
    print("[analyze_sentiment] 分析新闻情绪...")
    time.sleep(0.3)
    return {"news_sentiment": "正面:65% 中性:25% 负面:10%", "sources": ["news_api"]}

def compare_competitors(state: ResearchState) -> dict:
    print("[compare_competitors] 竞品对比...")
    time.sleep(0.3)
    return {"competitor_info": "市场份额第一，技术领先", "sources": ["competitor_db"]}

def gather_experts(state: ResearchState) -> dict:
    print("[gather_experts] 收集专家观点...")
    time.sleep(0.3)
    return {"expert_opinions": "多家机构给予买入评级", "sources": ["expert_network"]}

def synthesize_report(state: ResearchState) -> dict:
    print("[synthesize_report] 综合分析生成报告...")
    report = f"""研究报告: {state['topic']}
================================
1. 财务表现: {state['finance_data']}
2. 市场情绪: {state['news_sentiment']}
3. 竞争格局: {state['competitor_info']}
4. 专家观点: {state['expert_opinions']}
数据来源: {state['sources']}
    """.strip()
    return {"report": report}

builder = StateGraph(ResearchState)
builder.add_node("collect_finance", collect_finance)
builder.add_node("analyze_sentiment", analyze_sentiment)
builder.add_node("compare_competitors", compare_competitors)
builder.add_node("gather_experts", gather_experts)
builder.add_node("synthesize_report", synthesize_report)

# Fan-out: START 同时触发 4 个并行节点
builder.add_edge(START, "collect_finance")
builder.add_edge(START, "analyze_sentiment")
builder.add_edge(START, "compare_competitors")
builder.add_edge(START, "gather_experts")

# Fan-in: 4 个节点都完成后汇聚到 synthesize_report
builder.add_edge("collect_finance", "synthesize_report")
builder.add_edge("analyze_sentiment", "synthesize_report")
builder.add_edge("compare_competitors", "synthesize_report")
builder.add_edge("gather_experts", "synthesize_report")
builder.add_edge("synthesize_report", END)

research_graph = builder.compile()
print("ResearchForge 并行图编译成功！")
print("\n结构: START → [4个并行节点] → Fan-in → 汇总报告 → END")

In [ ]:
# 可视化图结构
try:
    from IPython.display import Image, display
    png_bytes = research_graph.get_graph().draw_mermaid_png()
    display(Image(data=png_bytes))
    print("上图展示了 Fan-out/Fan-in 并行结构")
except Exception as e:
    print(f"可视化失败（不影响功能）: {e}")
    print("可在支持的环境中查看图结构")

In [ ]:
# 执行并行图
print("=" * 60)
print("执行并行研报生成...")
print("=" * 60)

start_time = time.time()
result = research_graph.invoke({
    "topic": "某科技公司 Q2 分析",
    "finance_data": "", "news_sentiment": "", "competitor_info": "",
    "expert_opinions": "", "report": "", "sources": []
})
elapsed = time.time() - start_time

print(f"\n执行耗时: {elapsed:.2f} 秒")
print(f"（4 个并行节点各约 0.3 秒，串行需要约 1.2 秒）")
print("\n最终报告:")
print(result["report"])
print(f"\n数据来源: {result['sources']}")

### 1.3 执行模式对比

| 维度 | 线性执行 | 并行执行 |
|------|---------|---------|
| **执行顺序** | 严格顺序 | 同一 step 内无序 |
| **总耗时** | 所有节点时间之和 | 关键路径时间 |
| **状态更新** | 单一更新 | 需要 Reducer 合并 |
| **适用场景** | 有依赖关系 | 独立任务 |
| **性能提升** | 无 | 可显著提升（2-5x） |

```
串行耗时: T1 + T2 + T3 + ... + Tn
并行耗时: max(T1, T2, ..., Tk) + T(k+1) + ...
```

---

## 2. Reducer 机制

### 2.1 为什么需要 Reducer？

当多个并行节点同时修改同一个 State 字段时，如果没有 Reducer，就会「写覆盖」——最后一个完成的节点把前面节点的数据全盖掉了。

Reducer 是并行的「和平协议」——规定大家怎么合并结果。

### 2.2 使用 Annotated 指定 Reducer

In [ ]:
# Reducer 合并策略示例
print("=== Reducer 合并策略 ===\n")

print("State 定义:")
print("  sources: Annotated[list, add]")
print("\n4 个并行节点各自返回:")
print("  collect_finance    -> {'sources': ['finance_report']}")
print("  analyze_sentiment  -> {'sources': ['news_api']}")
print("  compare_competitors-> {'sources': ['competitor_db']}")
print("  gather_experts     -> {'sources': ['expert_network']}")
print(f"\n合并结果: sources = {result['sources']}")

print("\n=== 常用 Reducer 策略 ===")
reducers = [
    ("operator.add", "列表拼接 / 数值累加", "收集所有结果"),
    ("lambda old, new: new", "直接覆盖", "只保留最新值"),
    ("sorting_reducer", "合并并排序", "按优先级排序"),
    ("unique_reducer", "合并并去重", "实体提取场景"),
    ("keep_last_n(10)", "保留最近 N 个", "消息历史管理"),
]
for name, behavior, scene in reducers:
    print(f"  {name:<20} | {behavior:<12} | {scene}")

### 2.3 自定义 Reducer 示例

In [ ]:
# 自定义 Reducerdef sorting_reducer(left, right):    """合并并排序"""    if not isinstance(left, list):        left = [left]    if not isinstance(right, list):        right = [right]    return sorted(left + right)def unique_reducer(left, right):    """合并并去重"""(    combined = (left if isinstance(left, list) else [left]) +               (right if isinstance(right, list) else [right]))    return list(dict.fromkeys(combined))def keep_last_n(n=5):    """保留最近 N 个"""    def reducer(left, right):(        combined = (left if isinstance(left, list) else [left]) +                   (right if isinstance(right, list) else [right]))        return combined[-n:]    return reducer# 测试 Reducerprint("=== Reducer 测试 ===\n")print("sorting_reducer([3, 1], [2, 4]) =", sorting_reducer([3, 1], [2, 4]))print("unique_reducer(['a', 'b'], ['b', 'c']) =", unique_reducer(['a', 'b'], ['b', 'c']))print("keep_last_n(3)([1, 2], [3, 4, 5]) =", keep_last_n(3)([1, 2], [3, 4, 5]))print("\n=== 常见错误 ===")print("InvalidUpdateError: At key 'state': Can receive only one value per step.")print("原因: 多个并行节点更新同一键，但未指定 reducer。")print("解决: class State(TypedDict): state: Annotated[list, operator.add]")

---

## 3. Send 原语与动态 Map-Reduce

### 3.1 什么是 Map-Reduce？

Map-Reduce 是一种分布式计算模式：
- **Map 阶段**：将大任务拆分为多个子任务，并行执行
- **Reduce 阶段**：收集所有子任务的结果，汇总成最终结果

**类比**：Map-Reduce 像是一个老师批改作文——有 50 篇作文，分给 5 个助教，每人 10 篇，最后汇总成绩。`Send` 就是「分发」这个动作。

### 3.2 Send API

In [ ]:
from langgraph.types import Send

class MapReduceState(TypedDict):
    documents: list
    summaries: Annotated[list, add]
    final_summary: str

def map_documents(state: MapReduceState) -> list:
    """
    Map 阶段：动态分发每个文档到子任务。
    返回 Send 列表，每个 Send 触发一次子图执行。
    """
    print(f"[map] 分发 {len(state['documents'])} 个文档到子任务")
    return [
        Send("summarize_doc", {"doc": doc})
        for doc in state["documents"]
    ]

def summarize_doc(state: dict) -> dict:
    """处理单个文档的摘要"""
    doc = state["doc"]
    print(f"[summarize_doc] 处理文档: {doc[:30]}...")
    summary = f"摘要: {doc}"
    return {"summaries": [summary]}

def reduce_summaries(state: MapReduceState) -> dict:
    """Reduce 阶段：合并所有摘要"""
    print(f"[reduce] 合并 {len(state['summaries'])} 个摘要")
    return {"final_summary": " | ".join(state["summaries"])}

mr_builder = StateGraph(MapReduceState)
mr_builder.add_node("map_documents", map_documents)
mr_builder.add_node("summarize_doc", summarize_doc)
mr_builder.add_node("reduce_summaries", reduce_summaries)

# 条件边：map_documents 返回 Send 列表，动态分发
mr_builder.add_conditional_edges(START, map_documents, ["summarize_doc"])
# Fan-in：所有 summarize_doc 完成后汇聚到 reduce_summaries
mr_builder.add_edge("summarize_doc", "reduce_summaries")
mr_builder.add_edge("reduce_summaries", END)

mr_graph = mr_builder.compile()
print("Map-Reduce 图编译成功！")
print("\n执行流程:")
print("  START → map_documents → [Send x N] → summarize_doc(并行) → Fan-in → reduce_summaries → END")

In [ ]:
# 执行 Map-Reduce
from pathlib import Path
import json

print("=" * 60)
print("执行动态 Map-Reduce...")
print("=" * 60)

data_path = Path("map_reduce_documents.json")
if not data_path.exists():
    data_path = Path("turtorial/LG-05-subgraphs/map_reduce_documents.json")

documents = json.loads(data_path.read_text(encoding="utf-8"))["documents"]
print(f"已加载 {len(documents)} 篇课程文档")

mr_result = mr_graph.invoke({
    "documents": documents,
    "summaries": [],
    "final_summary": ""
})

print(f"\n最终摘要:")
print(mr_result["final_summary"])


### 3.3 Send 关键点

1. **Send 的第二个参数是子图的初始 State**，不是父图的 State
2. **子图执行时，State 从 Send 的参数开始**，独立于父图
3. **子图的结果通过公共字段写回父图**（需要 Reducer 合并）
4. **所有子图实例完成后，LangGraph 自动触发 Fan-in**

```
[输入: documents=[A, B, C]]
         ↓
   [map_documents 节点]
         ↓
   Send("summarize_doc", {doc: A})
   Send("summarize_doc", {doc: B})   ← 并行分发
   Send("summarize_doc", {doc: C})
         ↓
   ┌─────────────────────────────┐
   ↓              ↓              ↓
[子图:A]      [子图:B]       [子图:C]
         ↓              ↓              ↓
   └─────────────────────────────┘
         ↓
    Fan-in 汇聚
         ↓
   [reduce_summaries]
         ↓
   [输出: final_summary=...]
```

---

## 4. Subgraph：子图构建与集成

### 4.1 为什么需要子图？

- **模块化**：将复杂逻辑拆分为独立的模块
- **复用**：同一个子图可以嵌入多个不同的父图
- **隔离复杂度**：子图内部的中间状态不会污染父图
- **可测试性**：子图可以独立运行和调试

### 4.2 子图定义与嵌入

In [ ]:
# ========== 1. 定义子图 ==========
class FinanceState(TypedDict):
    """子图自己的 State——完全独立定义。"""
    ticker: str
    revenue: str
    profit: str

def fetch_revenue(state: FinanceState) -> dict:
    """模拟获取营收数据。"""
    return {"revenue": f"{state['ticker']} 营收: 100亿"}

def fetch_profit(state: FinanceState) -> dict:
    """模拟获取利润数据。"""
    return {"profit": f"{state['ticker']} 利润: 15亿"}

finance_builder = StateGraph(FinanceState)
finance_builder.add_node("fetch_revenue", fetch_revenue)
finance_builder.add_node("fetch_profit", fetch_profit)
finance_builder.add_edge(START, "fetch_revenue")
finance_builder.add_edge("fetch_revenue", "fetch_profit")
finance_builder.add_edge("fetch_profit", END)

finance_subgraph = finance_builder.compile()
print("财经数据子图编译成功！")
print("子图拥有独立的 FinanceState，与父图隔离")

# 子图可以独立测试
print("\n=== 子图独立测试 ===")
sub_result = finance_subgraph.invoke({"ticker": "AAPL", "revenue": "", "profit": ""})
print(f"子图输出: {sub_result}")

In [ ]:
# ========== 2. 定义父图并嵌入子图 ==========
class MainState(TypedDict):
    topic: str
    ticker: str
    finance_summary: str
    final_report: str

def get_ticker(state: MainState) -> dict:
    return {"ticker": "AAPL"}

def summarize_finance(state: MainState) -> dict:
    return {"finance_summary": f"{state['ticker']}: 营收100亿, 利润15亿"}

def generate_final(state: MainState) -> dict:
    return {"final_report": f"报告: {state['finance_summary']}"}

main_builder = StateGraph(MainState)
main_builder.add_node("get_ticker", get_ticker)
# 关键：把编译后的子图作为节点嵌入父图
main_builder.add_node("finance_subgraph", finance_subgraph)
main_builder.add_node("summarize_finance", summarize_finance)
main_builder.add_node("generate_final", generate_final)

main_builder.add_edge(START, "get_ticker")
main_builder.add_edge("get_ticker", "finance_subgraph")
main_builder.add_edge("finance_subgraph", "summarize_finance")
main_builder.add_edge("summarize_finance", "generate_final")
main_builder.add_edge("generate_final", END)

main_graph = main_builder.compile()
print("主图编译成功（包含子图节点）！")

In [ ]:
# 可视化主图结构
try:
    png_bytes = main_graph.get_graph().draw_mermaid_png()
    display(Image(data=png_bytes))
    print("上图展示了子图嵌入主图的结构")
except Exception as e:
    print(f"可视化失败: {e}")

# 执行主图
print("\n=== 执行主图 ===")
main_result = main_graph.invoke({
    "topic": "AI 芯片行业分析",
    "ticker": "",
    "finance_summary": "",
    "final_report": ""
})
print(f"最终报告: {main_result['final_report']}")

### 4.3 子图核心要点

- 子图是一个**独立的 `StateGraph`**，有自己完整的 State、Node、Edge 定义
- 子图必须先 `.compile()`，然后作为 action 传给父图的 `add_node`
- 子图可以单独测试：`subgraph.invoke({...})`
- 子图默认**继承父图的 checkpointer**（如果父图配置了的话）

---

## 5. 输入/输出映射

### 5.1 状态隔离：子图是「黑盒」

默认行为：子图的 State 与父图完全隔离。

```
父图 State: {topic, findings}     子图 State: {query, raw_data, summary}
              ↑                              ↑
         父图可见                        子图内部可见
         子图不可见                      父图不可见
```

**隔离的好处**：
1. **封装性**：子图内部的中间状态（`raw_data`）不会污染父图
2. **可复用性**：同一个子图可以嵌入多个不同的父图
3. **可测试性**：子图可以独立运行和调试

**隔离的代价**：父子图字段名不一致时，数据不会自动传递，必须显式配置映射关系。

### 5.2 input / output 映射配置

In [ ]:
# input/output 映射示例# 父图 Stateclass ParentState(TypedDict):    topic: str           # 父图叫 "topic"    findings: list       # 父图叫 "findings"# 子图 State（字段名完全不同）class ChildState(TypedDict):    query: str           # 子图叫 "query"    results: list        # 子图叫 "results"def child_fetch(state: ChildState) -> dict:    return {"results": [f"Result for {state.get('query', '')}"]}child_builder = StateGraph(ChildState)child_builder.add_node("fetch", child_fetch)child_builder.add_edge(START, "fetch")child_builder.add_edge("fetch", END)child_graph = child_builder.compile()# 父图parent_builder = StateGraph(ParentState)# 配置 input/output 映射# input:  父图 "topic" → 子图 "query"# output: 子图 "results" → 父图 "findings"parent_builder.add_node(    "research",    child_graph,    input=lambda state: {"query": state["topic"]},    output=lambda state: {"findings": state["results"]},)parent_builder.add_edge(START, "research")parent_builder.add_edge("research", END)mapped_graph = parent_builder.compile()print("带映射的图编译成功！")# 执行mapped_result = mapped_graph.invoke({"topic": "AI 芯片", "findings": []})print(f"结果: {mapped_result}")print("\n=== 映射的本质 ===")print("  input 函数 = '翻译器'：把父图数据格式翻译成子图能理解的格式")print("  output 函数 = '翻译器'：把子图结果格式翻译回父图的数据格式")

### 5.3 公共字段模式

当父子图有**相同字段名**时，不需要配置映射，字段自动透传。

```python
class SharedState(TypedDict):
    messages: list       # 父子图都有 "messages"
    user_id: str         # 父子图都有 "user_id"

def child_node(state: SharedState):
    # 子图可以直接读写父图的 "messages"
    return {"messages": state["messages"] + [{"role": "child", "content": "..."}]}

# 无需 input/output 映射——"messages" 和 "user_id" 自动共享
parent_builder.add_node("child", child_graph)
```

**共享字段的注意事项**：
- 优点：简单，不需要写映射函数
- 风险：子图可能意外修改父图的关键字段 → 建议子图只添加/追加，不要删除

### 5.4 跨子图通信的三种方式

| 方式 | 适用场景 | 代码示例 |
|------|---------|---------|
| **公共字段** | 父子图有相同字段名 | `messages` 在父子图中同名，自动透传 |
| **input/output 映射** | 字段名不同，需要格式转换 | `input=lambda s: {"query": s["topic"]}` |
| **Store（长期记忆）** | 多个子图需要共享数据 | `store.aput(user_id, "key", data)` |

```python
# Store 跨子图通信示例
def child_a(state, *, store):
    store.aput("shared", "intermediate_result", {"data": "..."})
    return {"status": "done"}

def child_b(state, *, store):
    result = store.aget("shared", "intermediate_result")
    return {"output": result}
```

---

## 6. Send + Subgraph 组合：动态分发子图

将 Send 的动态分发能力与 Subgraph 的模块化结合，实现更强大的并行处理。

In [ ]:
# ========== 1. 定义子图（每个公司的分析子图）==========
class CompanyState(TypedDict):
    company: str
    analysis: str

def analyze_company(state: CompanyState) -> dict:
    """分析单个公司。"""
    return {"analysis": f"{state['company']} 的分析报告：营收增长，前景良好"}

company_builder = StateGraph(CompanyState)
company_builder.add_node("analyze", analyze_company)
company_builder.add_edge(START, "analyze")
company_builder.add_edge("analyze", END)
company_graph = company_builder.compile()

# ========== 2. 定义父图 ==========
class ParentState(TypedDict):
    companies: list[str]
    reports: Annotated[list[str], add]   # Reducer 合并所有子图结果

def dispatch_companies(state: ParentState) -> list:
    """
    Map 阶段：动态分发每个公司到子图。
    返回 Send 列表，每个 Send 触发一次子图执行。
    """
    return [
        Send("analyze_company", {"company": c})
        for c in state["companies"]
    ]

def collect_reports(state: ParentState) -> dict:
    """Reduce 阶段：所有子图完成后汇聚。"""
    return {"reports": state["reports"]}

parent_builder = StateGraph(ParentState)
parent_builder.add_node("dispatch", dispatch_companies)
# 关键：子图作为节点，可以被 Send 动态触发
parent_builder.add_node("analyze_company", company_graph)
parent_builder.add_node("collect", collect_reports)

# 条件边：dispatch 返回 Send 列表，动态分发
parent_builder.add_conditional_edges(START, dispatch_companies, ["analyze_company"])
# Fan-in：所有 analyze_company 完成后汇聚到 collect
parent_builder.add_edge("analyze_company", "collect")
parent_builder.add_edge("collect", END)

parent_graph = parent_builder.compile()
print("Send + Subgraph 组合图编译成功！")

In [ ]:
# 执行动态分发子图
print("=" * 60)
print("执行动态 Map-Reduce（子图分发）...")
print("=" * 60)

companies = ["腾讯", "阿里", "字节", "美团", "京东"]

parent_result = parent_graph.invoke({
    "companies": companies,
    "reports": []
})

print(f"\n共生成 {len(parent_result['reports'])} 份分析报告:")
for i, report in enumerate(parent_result["reports"], 1):
    print(f"  {i}. {report}")

---

## 7. 并行模式最佳实践

### 7.1 何时使用并行执行？

**适用场景**：
- 多数据源检索（2-5x 提速）
- 多模型推理
- 批量独立任务
- 多智能体协作
- 多维度分析

**不适用场景**：
- 任务有依赖关系
- 共享资源冲突
- API 限流
- 内存受限
- 严格顺序要求

### 7.2 子图最佳实践 Checklist

1. **先独立测试子图**：`subgraph.invoke({...})` 确保子图自身逻辑正确
2. **再嵌入父图**：逐步集成，先不加映射，确认能跑通
3. **最后加映射**：配置 input/output，处理字段名不一致的情况
4. **优先用公共字段**：如果父子图可以用相同字段名，就不要写映射函数
5. **子图 State 要最小化**：只放子图真正需要的字段，不要复制整个父图 State

### 7.3 状态设计最佳实践

In [ ]:
# 状态设计最佳实践

class BestPracticeState(TypedDict):
    # 输入字段 (只读)
    question: str
    user_id: str
    
    # 并行收集的数据 (使用 reducer)
    search_results: Annotated[list, add]
    
    # 最终输出 (单一节点更新)
    answer: str

print("=== 状态设计最佳实践 ===\n")
print("1. 输入字段: 只读，由初始 state 提供")
print("2. 并行收集字段: 使用 Annotated + Reducer")
print("3. 最终输出字段: 单一节点更新，无需 Reducer")

print("\n=== 嵌套子图状态扁平化 ===")
print("不推荐: 深层嵌套状态")
print("  class DeepState(TypedDict):")
print("      level1: {'level2': {'level3': {'data': ...}}}")
print("\n推荐: 扁平化")
print("  class FlatState(TypedDict):")
print("      news_results: list      # 新闻子图的结果")
print("      finance_results: list   # 财经子图的结果")
print("      final_report: str       # 汇总结果")

---

## 8. 并发控制与超时

### 8.1 控制并发数

当 Send 分发大量子任务时，需要限制同时执行的并发数，避免资源耗尽。

In [ ]:
import asyncio

# 并发控制示例
semaphore = asyncio.Semaphore(3)  # 最多 3 个并发

async def limited_search(state, max_concurrent=3):
    """
    带并发限制的搜索。
    使用信号量限制同时执行的子任务数。
    """
    sem = asyncio.Semaphore(max_concurrent)
    
    async def search_one(query):
        async with sem:
            # 模拟搜索
            await asyncio.sleep(0.1)
            return f"结果: {query}"
    
    queries = state.get("queries", [])
    tasks = [search_one(q) for q in queries]
    results = await asyncio.gather(*tasks)
    return {"results": results}

print("并发控制函数已定义")
print("\n使用 asyncio.Semaphore(max_concurrent) 限制并发数")
print("避免 Send 分发太多子任务导致内存/连接池耗尽")

### 8.2 超时控制

In [ ]:
async def search_with_timeout(state, timeout=5.0):
    """
    带超时的搜索。
    如果搜索超过 timeout 秒，返回超时标记。
    """
    try:
        return await asyncio.wait_for(
            actual_search(state),
            timeout=timeout
        )
    except asyncio.TimeoutError:
        print(f"[timeout] 搜索超时 ({timeout}s)")
        return {"results": ["<Timeout>"]}

async def actual_search(state):
    """模拟搜索（可能很慢）"""
    await asyncio.sleep(0.1)
    return {"results": ["搜索成功"]}

print("超时控制函数已定义")
print("\n使用 asyncio.wait_for(coro, timeout) 设置超时")
print("超时后返回降级结果，避免阻塞整个图")

### 8.3 错误处理包装器

In [ ]:
def safe_wrapper(func):
    """
    错误处理包装器。
    捕获异常并返回降级结果，避免单个节点失败导致整个图失败。
    """
    def wrapper(state):
        try:
            return func(state)
        except Exception as e:
            print(f"[error] {func.__name__} 失败: {e}")
            return {"results": [f"<Error: {str(e)}>"]}
    return wrapper

# 使用包装器
def risky_search(state):
    """可能失败的搜索"""
    if state.get("should_fail"):
        raise RuntimeError("模拟搜索失败")
    return {"results": ["搜索成功"]}

safe_search = safe_wrapper(risky_search)

print("=== 错误处理测试 ===")
print("\n正常情况:")
print(safe_search({"should_fail": False}))

print("\n失败情况:")
print(safe_search({"should_fail": True}))

print("\n错误处理包装器确保单个节点失败不会导致整个图失败")

### 8.4 调试技巧

In [ ]:
import time
from functools import wraps

def trace_execution(func):
    """
    执行追踪装饰器。
    打印节点的开始/结束时间和耗时。
    """
    @wraps(func)
    def wrapper(state):
        start = time.time()
        print(f"🚀 开始: {func.__name__}")
        result = func(state)
        elapsed = time.time() - start
        print(f"✅ 完成: {func.__name__} ({elapsed:.2f}s)")
        return result
    return wrapper

# 应用到节点
@trace_execution
def traced_node(state):
    time.sleep(0.2)
    return {"result": "done"}

print("=== 追踪测试 ===")
traced_node({})

print("\n=== 调试技巧总结 ===")
print("1. 逐个节点验证：先不并行，串行跑通每个节点，再改成并行")
print("2. Reducer 验证：打印合并前后的 State，确认没有数据丢失")
print("3. 子图单独测试：每个子图作为独立图先跑通，再嵌入父图")
print("4. 使用 trace_execution 追踪节点执行时间")

---

## 9. 完整案例：ResearchForge（并行研报生成）

综合使用 Fan-out/Fan-in、Reducer、Send、Subgraph，构建一个并行研报生成系统。

In [ ]:
# ResearchForge 完整示例

# === 子图 1: 财经数据子图 ===
class FinanceSubState(TypedDict):
    ticker: str
    revenue: str
    profit_margin: str
    finance_score: float

def fetch_finance_data(state: FinanceSubState) -> dict:
    time.sleep(0.2)
    return {
        "revenue": "100亿",
        "profit_margin": "15%",
        "finance_score": 85.0
    }

finance_sg = StateGraph(FinanceSubState)
finance_sg.add_node("fetch", fetch_finance_data)
finance_sg.add_edge(START, "fetch")
finance_sg.add_edge("fetch", END)
finance_graph = finance_sg.compile()

# === 子图 2: 新闻舆情子图 ===
class NewsSubState(TypedDict):
    keyword: str
    sentiment: str
    hot_topics: list
    news_score: float

def fetch_news_data(state: NewsSubState) -> dict:
    time.sleep(0.2)
    return {
        "sentiment": "正面",
        "hot_topics": ["AI投资", "新产品发布"],
        "news_score": 78.0
    }

news_sg = StateGraph(NewsSubState)
news_sg.add_node("fetch", fetch_news_data)
news_sg.add_edge(START, "fetch")
news_sg.add_edge("fetch", END)
news_graph = news_sg.compile()

# === 父图 ===
class ResearchForgeState(TypedDict):
    topic: str
    ticker: str
    keyword: str
    finance_result: dict
    news_result: dict
    final_score: float
    report: str

def parse_topic(state: ResearchForgeState) -> dict:
    return {
        "ticker": "AAPL",
        "keyword": state["topic"]
    }

def compile_final_report(state: ResearchForgeState) -> dict:
    finance = state.get("finance_result", {})
    news = state.get("news_result", {})
    score = (finance.get("finance_score", 0) + news.get("news_score", 0)) / 2
    
    report = f"""
=== {state['topic']} 研报 ===
财务评分: {finance.get('finance_score', 0)}
舆情评分: {news.get('news_score', 0)}
综合评分: {score:.1f}

财务数据: 营收 {finance.get('revenue', 'N/A')}, 利润率 {finance.get('profit_margin', 'N/A')}
舆情概况: {news.get('sentiment', 'N/A')}, 热点: {news.get('hot_topics', [])}
""".strip()
    
    return {"final_score": score, "report": report}

rf_builder = StateGraph(ResearchForgeState)
rf_builder.add_node("parse", parse_topic)

# 嵌入子图（带 input/output 映射）
rf_builder.add_node(
    "finance_analysis",
    finance_graph,
    input=lambda s: {"ticker": s["ticker"]},
    output=lambda s: {"finance_result": dict(s)},
)
rf_builder.add_node(
    "news_analysis",
    news_graph,
    input=lambda s: {"keyword": s["keyword"]},
    output=lambda s: {"news_result": dict(s)},
)

rf_builder.add_node("compile_report", compile_final_report)

# Fan-out: parse 后并行执行两个子图
rf_builder.add_edge(START, "parse")
rf_builder.add_edge("parse", "finance_analysis")
rf_builder.add_edge("parse", "news_analysis")

# Fan-in: 两个子图都完成后汇总
rf_builder.add_edge("finance_analysis", "compile_report")
rf_builder.add_edge("news_analysis", "compile_report")
rf_builder.add_edge("compile_report", END)

research_forge = rf_builder.compile()
print("ResearchForge 编译成功！")
print("结构: START → parse → [finance_analysis || news_analysis] → compile_report → END")

In [ ]:
# 执行 ResearchForge
print("=" * 60)
print("执行 ResearchForge 并行研报生成...")
print("=" * 60)

start = time.time()
rf_result = research_forge.invoke({
    "topic": "某科技公司 Q2 分析",
    "ticker": "",
    "keyword": "",
    "finance_result": {},
    "news_result": {},
    "final_score": 0.0,
    "report": ""
})
elapsed = time.time() - start

print(f"\n执行耗时: {elapsed:.2f} 秒")
print(f"综合评分: {rf_result['final_score']:.1f}")
print("\n研报内容:")
print(rf_result["report"])

---

## 10. 常见误区与注意事项

| 坑 | 现象 | 解决方案 |
|----|------|---------|
| **并行节点修改同一个非 Reducer 字段** | 数据互相覆盖 | 并行节点共享的字段必须用 `Annotated + Reducer` |
| **Fan-in 节点前置有循环** | Fan-in 触发条件不明确 | 确保循环有明确终止条件 |
| **子图 State 与父图不兼容** | 字段映射错误，数据传不过去 | 子图的 input/output 映射必须显式配置 |
| **Send 分发太多子任务** | 内存/连接池耗尽 | 使用 `max_concurrency` 限制同时执行的子任务数 |
| **递归限制不当** | 并行 + 循环导致超级步过多 | 生产环境合理设置 `recursion_limit` |
| **更新顺序不可控** | 并行节点的更新顺序无法预测 | 使用自定义排序 Reducer 或临时字段 + 汇聚节点 |
| **并行节点没有真正并发** | 节点仍然串行执行 | 使用异步函数 `async def` |

### 调试技巧
1. **逐个节点验证**：先不并行，串行跑通每个节点，再改成并行
2. **Reducer 验证**：打印合并前后的 State，确认没有数据丢失
3. **子图单独测试**：每个子图作为独立图先跑通，再嵌入父图

---

## 11. 速查表 / Cheat Sheet

### 并行图结构速查

```python
# 简单并行
builder.add_edge(START, "task1")
builder.add_edge(START, "task2")
builder.add_edge(["task1", "task2"], "merge")

# 不同长度路径
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "b2")
builder.add_edge(["b2", "c"], "d")

# Send 动态分发
from langgraph.types import Send

def dispatch(state):
    return [Send("target", {"key": val}) for val in state["items"]]

builder.add_conditional_edges(START, dispatch, ["target"])
builder.add_edge("target", "collect")
```

### Reducer 速查

```python
import operator

# 列表拼接
Annotated[list, operator.add]

# 数值求和
Annotated[int, operator.add]

# 只保留最新
Annotated[str, lambda l, r: r]

# 保留最近 N 个
def keep_last_n(n):
    return lambda l, r: ((l if isinstance(l, list) else [l]) + \
                        (r if isinstance(r, list) else [r]))[-n:]

Annotated[list, keep_last_n(10)]
```

### 子图速查

```python
# 1. 定义子图
child_builder = StateGraph(ChildState)
child_builder.add_node(...)
child_graph = child_builder.compile()

# 2. 嵌入父图（无映射）
parent_builder.add_node("child", child_graph)

# 3. 嵌入父图（带映射）
parent_builder.add_node(
    "child",
    child_graph,
    input=lambda s: {"child_key": s["parent_key"]},
    output=lambda s: {"parent_key": s["child_key"]},
)

# 4. 子图独立测试
child_graph.invoke({"key": "value"})
```

### 并发控制速查

```python
import asyncio

# 信号量限制并发
sem = asyncio.Semaphore(3)
async def limited_task():
    async with sem:
        return await do_work()

# 超时控制
try:
    result = await asyncio.wait_for(task(), timeout=5.0)
except asyncio.TimeoutError:
    result = {"error": "timeout"}

# 错误包装
def safe_wrapper(func):
    def wrapper(state):
        try:
            return func(state)
        except Exception as e:
            return {"error": str(e)}
    return wrapper
```

### 记忆口诀

> **"Fan-out 是群发，Fan-in 是等齐人；Reducer 是合并器，Send 是动态派活，Subgraph 是模块包。"**

---

## 12. 课后练习

### 练习 1：多源并行检索问答系统

同时从 Wikipedia 和 Web 搜索获取信息生成答案：
- 定义 `search_web` 和 `search_wikipedia` 两个并行节点
- 使用 `Annotated[list, operator.add]` 合并搜索结果
- `generate_answer` 节点汇聚结果生成最终答案
- 对比串行和并行的执行耗时

### 练习 2：为 ResearchForge 增加结果规范化子图

- 创建一个专门规范化检索结果的子图（去重、格式化、过滤）
- 子图有自己的 State（`findings`, `normalized_findings`）
- 使用 input/output 映射将父图的 `findings` 传入子图
- 子图输出 `normalized_findings` 回到父图

### 练习 3：带 max_concurrency 的并行图

- 使用 Send 动态分发 10 个文档处理任务
- 使用 `asyncio.Semaphore(3)` 限制同时执行的并发数为 3
- 每个任务添加随机耗时（0.1-0.5 秒）
- 观察并发限制的效果

### 练习 4：使用 Send 实现文档批量处理

- 输入：一个包含 20 个文档的列表
- Map 阶段：使用 Send 将每个文档分发到 `process_doc` 节点
- 每个 `process_doc` 节点：提取关键词、生成摘要
- Reduce 阶段：合并所有摘要，生成最终报告
- 使用 Reducer 确保结果不丢失

### 练习 5：子图嵌套与状态扁平化

- 创建一个三层嵌套的子图结构：父图 → 分析子图 → 数据获取子图
- 确保各层 State 扁平化，不使用嵌套 dict
- 使用公共字段和 input/output 映射组合传递数据
- 测试各层子图的独立执行

---

**下一节**: LG-06 预构建 Agent